In [5]:
# Import all required modules for EDA and visualization
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns # type: ignore
from statsmodels.tsa.seasonal import seasonal_decompose # type: ignore
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf # type: ignore

import matplotlib.pyplot as plt
plt.style.use('seaborn-v0_8-whitegrid')   # Pick any style from plt.style.available



sns.set_context("talk")


In [6]:
import os
import sys
import shutil
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns # type: ignore
import datetime
from statsmodels.tsa.seasonal import seasonal_decompose # type: ignore
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf # type: ignore
import traceback

In [7]:
# --- CHANGE THESE AS NEEDED ---
INFILE = r"C:\Users\samru\OneDrive\Desktop\SIH_2025\AI-Pollution-Forecast-and-Policy-Dashboard\ML\Training_Data\Full_2023_DATASET.csv"   # <--- Change to your actual CSV file/path
OUT_CLEAN = "eda_cleaned_model_ready_2023.csv"
PLOTS_DIR = "eda_plots_2023"

# Create output directory for plots
os.makedirs(PLOTS_DIR, exist_ok=True)


In [8]:
def clean_col(c):
    c2 = str(c)
    # remove unit strings / broken-encoding fragments, keep '.' for decimals
    for target in ["(Ã‚Âµg/mÃ‚Â³)","(µg/m³)","(ppb)","(mg/m³)","(%)"]:
        c2 = c2.replace(target,"")
    # replace separators with underscore but keep '.' for names like pm2.5
    for ch in ['(',')','/','\\','%','-','·','Ã‚','Â','µ','³']:
        c2 = c2.replace(ch,'')
    c2 = c2.replace('  ',' ').strip()
    c2 = c2.replace(' ','_').lower()
    # normalize multiple underscores
    while '__' in c2:
        c2 = c2.replace('__','_')
    return c2.strip('_')

def save_fig(fig, name):
    # Saves matplotlib figure as PNG in output directory
    path = os.path.join(PLOTS_DIR, name)
    fig.savefig(path, bbox_inches='tight', dpi=150)
    print("Saved:", path)


In [9]:
print("Loading:", INFILE)
df = pd.read_csv(INFILE, low_memory=False)  # Load your data
df.columns = [clean_col(c) for c in df.columns]  # Clean column names
print("Columns after cleaning:", df.columns)


Loading: C:\Users\samru\OneDrive\Desktop\SIH_2025\AI-Pollution-Forecast-and-Policy-Dashboard\ML\Training_Data\Full_2023_DATASET.csv
Columns after cleaning: Index(['date', 'station', 'aqi', 'pm2.5_gm', 'pm10_gm', 'no_gm', 'no2_gm',
       'nox', 'nh3_gm', 'so2_gm', 'co_mgm', 'ozone_gm', 'benzene_gm',
       'toluene_gm', 'xylene_gm', 'pm2.5_gm_lag1', 'pm10_gm_lag1',
       'no2_gm_lag1', 'pm2.5_gm_lag3', 'pm10_gm_lag3', 'no2_gm_lag3',
       'pm2.5_gm_lag7', 'pm10_gm_lag7', 'no2_gm_lag7', 'pm2.5_gm_roll3d',
       'no2_gm_roll3d', 'pm2.5_gm_roll7d', 'fire_count', 'fire_intensity_sum',
       'fire_near_delhi_count', 'fire_3_day_sum', 'fire_7_day_sum',
       'is_tropomi_available', 'season', 'temp_mean', 'temp_min', 'temp_max',
       'humidity_mean', 'wind_mean', 'wind_max', 'blh_mean', 'pressure_mean',
       'aqi_lag1', 'fire_count_lag1', 'temp_mean_lag1', 'humidity_mean_lag1',
       'aqi_lag3', 'fire_count_lag3', 'temp_mean_lag3', 'humidity_mean_lag3',
       'aqi_lag7', 'fire_coun

In [10]:
# Attempts to create a 'datetime' column for indexing (must have time series info!)
if 'date' in df.columns:
    df['datetime'] = pd.to_datetime(df['date'], dayfirst=True, errors='coerce')
    df = df.drop(columns=['date'])
elif 'datetime' in df.columns:
    df['datetime'] = pd.to_datetime(df['datetime'], errors='coerce')
else:
    alt = [c for c in df.columns if 'time' in c or 'date' in c]
    if alt:
        df['datetime'] = pd.to_datetime(df[alt[0]], errors='coerce')
        df = df.drop(columns=[alt[0]])
    else:
        raise RuntimeError("No datetime column found. Rename your datetime or date column as 'date' or 'datetime'.")

df = df.sort_values('datetime').reset_index(drop=True)
df = df.set_index('datetime')
print("Shape after load:", df.shape)


Shape after load: (14235, 59)


In [11]:
# Numeric columns detection and smart gap filling/interpolation
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

# Interpolate small gaps first (limit=6 means fill up to 6 consecutive missing, else fallback)
df[numeric_cols] = df[numeric_cols].interpolate(limit=6, limit_direction='both')
df[numeric_cols] = df[numeric_cols].fillna(method='ffill').fillna(method='bfill')


C:\Users\samru\AppData\Local\Temp\ipykernel_9368\2510039550.py:6: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df[numeric_cols] = df[numeric_cols].fillna(method='ffill').fillna(method='bfill')


In [12]:
if 'aqi' not in df.columns:
    raise RuntimeError("No 'aqi' column found. Rename your AQI column to 'aqi'.")

df = df[~df['aqi'].isna()]  # Remove rows missing AQI target

# Add features for time-of-day effects
df['hour'] = df.index.hour
df['dayofweek'] = df.index.dayofweek
df['month'] = df.index.month
df['is_weekend'] = df['dayofweek'].isin([5,6]).astype(int)


In [13]:
df.reset_index().to_csv(OUT_CLEAN, index=False)
print("Saved cleaned EDA dataset:", OUT_CLEAN)


Saved cleaned EDA dataset: eda_cleaned_model_ready_2023.csv


In [14]:
missing = df.isna().sum().sort_values(ascending=False)
missing_percent = (missing / len(df) * 100).round(2)
missing_report = pd.concat([missing, missing_percent], axis=1)
missing_report.columns = ['missing_count', 'missing_percent']
missing_report.to_csv("missing_report.csv")
print("Missingness report saved: missing_report.csv")


Missingness report saved: missing_report.csv


In [15]:
fig, ax = plt.subplots(figsize=(14,4))
df['aqi'].plot(ax=ax)
ax.set_title("AQI — Time Series")
ax.set_ylabel("AQI")
save_fig(fig, "aqi_timeseries.png")
plt.close(fig)


Saved: eda_plots_2023\aqi_timeseries.png


In [16]:
fig, ax = plt.subplots(figsize=(10,4))
df.groupby('month')['aqi'].mean().plot(kind='bar', ax=ax)
ax.set_title("Monthly Average AQI")
ax.set_xlabel("Month")
ax.set_ylabel("AQI")
save_fig(fig, "monthly_avg_aqi.png")
plt.close(fig)


Saved: eda_plots_2023\monthly_avg_aqi.png


In [17]:
# Hourly pattern
fig, ax = plt.subplots(figsize=(10,4))
df.groupby('hour')['aqi'].mean().plot(ax=ax)
ax.set_title("Average AQI by Hour of Day")
ax.set_xlabel("Hour")
ax.set_ylabel("AQI")
save_fig(fig, "hourly_pattern_aqi.png")
plt.close(fig)

# Day of week pattern
fig, ax = plt.subplots(figsize=(8,4))
df.groupby('dayofweek')['aqi'].mean().plot(kind='bar', ax=ax)
ax.set_title("Average AQI by Day of Week")
save_fig(fig, "dayofweek_aqi.png")
plt.close(fig)


Saved: eda_plots_2023\hourly_pattern_aqi.png
Saved: eda_plots_2023\dayofweek_aqi.png


In [18]:
fig, ax = plt.subplots(figsize=(10,4))
sns.histplot(df['aqi'], bins=60, kde=True, ax=ax)
ax.set_title("AQI Distribution")
save_fig(fig, "aqi_distribution.png")
plt.close(fig)


Saved: eda_plots_2023\aqi_distribution.png


In [19]:
common_pollutants = ['pm2.5','pm25','pm10','no2','co','ozone','so2','nox','nh3','benzene']
detected = []
for key in common_pollutants:
    matches = [c for c in df.columns if key in c]
    if matches:
        detected.append(matches[0])
print("Detected pollutant-like columns:", detected)

# Pairplot for main variables (sample up to 2000 rows for speed)
vars_for_pairs = ['aqi'] + detected[:5]
if len(vars_for_pairs) > 1:
    sns.pairplot(df[vars_for_pairs].sample(min(2000, len(df))), plot_kws={'alpha':0.2})
    plt.suptitle("Pairplot (sample)", y=1.02)
    plt.savefig(os.path.join(PLOTS_DIR, "pairplot_pollutants_sample.png"), bbox_inches='tight', dpi=150)
    plt.close()


Detected pollutant-like columns: ['pm2.5_gm', 'pm10_gm', 'no2_gm', 'co_mgm', 'ozone_gm', 'so2_gm', 'nox', 'nh3_gm', 'benzene_gm']


In [20]:
for p in detected:
    try:
        fig, axs = plt.subplots(1,2, figsize=(14,4))
        sns.scatterplot(x=df[p].values, y=df['aqi'].values, alpha=0.4, ax=axs[0])
        axs[0].set_xlabel(p); axs[0].set_ylabel("AQI"); axs[0].set_title(f"AQI vs {p}")

        df[p].rolling(window=24, min_periods=1).mean().plot(ax=axs[1])
        axs[1].set_title(f"{p} — 24-hr rolling mean")
        save_fig(fig, f"{p}_vs_aqi_and_roll24.png")
        plt.close(fig)
    except Exception as e:
        print("Skipped plotting for", p, "due to", e)


Saved: eda_plots_2023\pm2.5_gm_vs_aqi_and_roll24.png
Saved: eda_plots_2023\pm10_gm_vs_aqi_and_roll24.png
Saved: eda_plots_2023\no2_gm_vs_aqi_and_roll24.png
Saved: eda_plots_2023\co_mgm_vs_aqi_and_roll24.png
Saved: eda_plots_2023\ozone_gm_vs_aqi_and_roll24.png
Saved: eda_plots_2023\so2_gm_vs_aqi_and_roll24.png
Saved: eda_plots_2023\nox_vs_aqi_and_roll24.png
Saved: eda_plots_2023\nh3_gm_vs_aqi_and_roll24.png
Saved: eda_plots_2023\benzene_gm_vs_aqi_and_roll24.png


In [22]:



 # Create 24-hr rolling sum for fire_count column (or similar)
fire_rolling_sum = df[fire_cols[0]].rolling(24).sum().reset_index(drop=True)
aqi_vals = df['aqi'].reset_index(drop=True)

# Plot AQI vs 24-hr fire count rolling sum (only where both are not NaN)
is_valid = ~fire_rolling_sum.isna() & ~aqi_vals.isna()
fig, ax = plt.subplots(figsize=(10,4))
sns.scatterplot(x=fire_rolling_sum[is_valid], y=aqi_vals[is_valid], alpha=0.4, ax=ax)
ax.set_title("AQI vs 24-hr summed fire_count (example)")
save_fig(fig, "aqi_vs_firecount24_scatter.png")
plt.close(fig)



Saved: eda_plots_2023\aqi_vs_firecount24_scatter.png


In [23]:
pivot = df.pivot_table(values='aqi', index=df.index.hour, columns=df.index.month, aggfunc='mean')
fig, ax = plt.subplots(figsize=(12,6))
sns.heatmap(pivot, cmap='viridis', ax=ax)
ax.set_title("AQI Heatmap — Hour vs Month")
save_fig(fig, "heatmap_hour_month.png")
plt.close(fig)


Saved: eda_plots_2023\heatmap_hour_month.png


In [24]:
fig = plt.figure(figsize=(12,5))
plot_acf(df['aqi'].dropna(), lags=72, ax=fig.add_subplot(121))
plot_pacf(df['aqi'].dropna(), lags=72, ax=fig.add_subplot(122))
plt.suptitle("ACF & PACF for AQI (up to 72 lags)")
save_fig(plt.gcf(), "acf_pacf_aqi.png")
plt.close()


Saved: eda_plots_2023\acf_pacf_aqi.png


In [25]:
try:
    daily = df['aqi'].resample('D').mean()
    result = seasonal_decompose(daily.dropna(), model='additive',
                                period=365//12 if len(daily)>400 else 7)
    fig = result.plot()
    fig.set_size_inches(12,8)
    save_fig(fig, "seasonal_decompose_aqi.png")
    plt.close(fig)
except Exception as e:
    print("Seasonal decompose skipped/failed:", e)


Saved: eda_plots_2023\seasonal_decompose_aqi.png


In [26]:
worst = df['aqi'].resample('D').mean().sort_values(ascending=False).head(10)
fig, ax = plt.subplots(figsize=(10,4))
worst.plot(kind='bar', color='crimson', ax=ax)
ax.set_title("Top 10 Worst AQI Days (daily mean)")
save_fig(fig, "top10_worst_days.png")
plt.close(fig)


Saved: eda_plots_2023\top10_worst_days.png


In [27]:
df['aqi_volatility_24h'] = df['aqi'].rolling(24).std()
fig, ax = plt.subplots(figsize=(12,4))
df['aqi_volatility_24h'].plot(ax=ax)
ax.set_title("AQI Volatility (24h rolling std)")
save_fig(fig, "aqi_volatility_24h.png")
plt.close(fig)


Saved: eda_plots_2023\aqi_volatility_24h.png


In [28]:
num = df.select_dtypes(include=[np.number]).copy()
corr = num.corr()
fig, ax = plt.subplots(figsize=(14,12))
sns.heatmap(corr, cmap='coolwarm', vmin=-1, vmax=1, ax=ax)
ax.set_title("Correlation matrix (numeric features)")
save_fig(fig, "correlation_heatmap.png")
plt.close(fig)

if 'aqi' in corr.columns:
    top_corr = corr['aqi'].abs().sort_values(ascending=False).head(40)
    top_corr.to_csv("top_corr_with_aqi.csv")
    print("Top correlations saved: top_corr_with_aqi.csv")


Saved: eda_plots_2023\correlation_heatmap.png
Top correlations saved: top_corr_with_aqi.csv
